<a href="https://colab.research.google.com/github/BaYa06/Final_project_GENAI/blob/main/GenAI_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("muraraimbekov/bishkek-real-estate-2025")

print("Path to dataset files:", path)

100%|██████████| 2.77M/2.77M [00:00<00:00, 46.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/muraraimbekov/bishkek-real-estate-2025/versions/10


In [2]:
import glob, re, random
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)

df = pd.read_csv(glob.glob(f"{path}/**/bishkek_apartments.csv", recursive=True)[0])
print(df.shape)

(8821, 60)


In [3]:
pats = [re.compile(p) for p in (
    r"(?<!\d)\+?996(?:[\s\-.()]{0,2}\d){9}(?!\d)",
    r"(?<!\d)\+?7(?:[\s\-.()]{0,2}\d){10}(?!\d)",
    r"(?<!\d)0(?:[\s\-.()]{0,2}\d){9}(?!\d)",
    r"(?<!\d)[2579](?:[\s\-.()]?\d){8}(?!\d)",
)]
def mask(s):
    for p in pats:
        s = p.sub("[PHONE]", s)
    return s

df["description_clean"] = df["description"].fillna("").map(mask)

In [4]:
# 1) убираем служебные и почти пустые колонки, а также сырой текст и адрес
service = ["id","listing_id","url","source","parsed_at","updated_at","photos_downloaded",
           "price_local","address","description"]
sparse = df.columns[df.isna().mean() > 0.95].tolist()
df = df.drop(columns=[c for c in service + sparse if c in df.columns])

# 2) без целевой колонки строка не нужна
df = df.dropna(subset=["price_per_m2"])

# 3) дубликаты (одинаковое описание, район, площадь, этаж)
before = len(df)
df = df.drop_duplicates(subset=["description_clean","district","area","floor"])
print("Удалено дублей:", before - len(df))

# 4) выбросы по цене за м²: оставляем 1–99 процентиль
lo, hi = df["price_per_m2"].quantile([0.01, 0.99])
df = df[df["price_per_m2"].between(lo, hi)]
print(f"Диапазон цены: {lo:.0f}–{hi:.0f}, строк осталось: {len(df)}")

# 5) смотрим значения меток для NLP
print(df["offer_type"].value_counts(dropna=False))
print(df["condition"].value_counts(dropna=False))

Удалено дублей: 174
Диапазон цены: 871–2624, строк осталось: 8472
offer_type
от агента          7296
от собственника    1166
NaN                  10
Name: count, dtype: int64
condition
евроремонт               3380
под самоотделку (псо)    2784
хорошее                  1178
NaN                       621
среднее                   349
не достроено              160
Name: count, dtype: int64


In [5]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2, random_state=SEED)
train.to_csv("train.csv", index=False); test.to_csv("test.csv", index=False)
print(len(train), len(test))

6777 1695


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

TARGET = "price_per_m2"
# price_usd — утечка (из него считается цель), текст и сырые колонки не берём
drop = {TARGET, "price_usd", "description_clean", "offer_type"} | {c for c in df.columns if c.startswith("raw_")}
feat = [c for c in train.columns if c not in drop]

num_cols = [c for c in feat if pd.api.types.is_numeric_dtype(train[c]) and not pd.api.types.is_bool_dtype(train[c])]
cat_cols = [c for c in feat if c not in num_cols and train[c].nunique() <= 60]
print("числовые:", num_cols); print("категориальные:", cat_cols)

def prep(d):
    d = d[num_cols + cat_cols].copy()
    d[cat_cols] = d[cat_cols].astype(str)
    return d

Xtr, Xte = prep(train), prep(test)
ytr, yte = train[TARGET], test[TARGET]

# 1) baseline: медиана по району
med = train.groupby("district")[TARGET].median()
base = test["district"].map(med).fillna(ytr.median())
print(f"Baseline (медиана по району): MAE = {mean_absolute_error(yte, base):.0f} $/м²")

pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), cat_cols),
])

# 2) линейная модель, 3) градиентный бустинг
models = {
    "Ridge": Ridge(alpha=1.0),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=SEED),
}
for name, m in models.items():
    pipe = Pipeline([("pre", pre), ("m", m)]).fit(Xtr, ytr)
    print(f"{name}: MAE = {mean_absolute_error(yte, pipe.predict(Xte)):.0f} $/м²")

числовые: ['residential_complex_id', 'rooms', 'area', 'living_area', 'kitchen_area', 'floor', 'total_floors', 'year_built', 'ceiling_height', 'latitude', 'longitude', 'is_active', 'photo_count', 'jk_parking', 'jk_gym']
категориальные: ['house_type', 'building_series', 'condition', 'bathroom', 'balcony', 'parking', 'furniture', 'floor_type', 'heating', 'internet', 'gas', 'has_phone', 'entrance_door', 'installment', 'mortgage', 'exchange_possible', 'jk_class', 'jk_status']
Baseline (медиана по району): MAE = 239 $/м²
Ridge: MAE = 208 $/м²
HistGradientBoosting: MAE = 133 $/м²


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

def run_nlp(label, threshold=0.6):
    tr = train.dropna(subset=[label]); te = test.dropna(subset=[label])
    clf = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True)),
        ("lr", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
    ]).fit(tr["description_clean"], tr[label])
    pred = clf.predict(te["description_clean"])
    print(f"\n=== {label} ===")
    print(classification_report(te[label], pred, zero_division=0))
    conf = clf.predict_proba(te["description_clean"]).max(axis=1)
    print(f"Доля «не уверен» при пороге {threshold}: {(conf < threshold).mean():.1%}")
    return clf

clf_offer = run_nlp("offer_type")
clf_cond = run_nlp("condition")


=== offer_type ===
                 precision    recall  f1-score   support

      от агента       0.95      0.85      0.89      1474
от собственника       0.39      0.67      0.49       219

       accuracy                           0.82      1693
      macro avg       0.67      0.76      0.69      1693
   weighted avg       0.87      0.82      0.84      1693

Доля «не уверен» при пороге 0.6: 12.2%

=== condition ===
                       precision    recall  f1-score   support

           евроремонт       0.83      0.68      0.75       671
         не достроено       0.04      0.30      0.07        27
под самоотделку (псо)       0.90      0.81      0.85       565
              среднее       0.26      0.35      0.29        55
              хорошее       0.47      0.42      0.45       252

             accuracy                           0.67      1570
            macro avg       0.50      0.51      0.48      1570
         weighted avg       0.77      0.67      0.71      1570

Доля «н

In [8]:
train = train.copy(); test = test.copy()
group = {
    "евроремонт": "евроремонт",
    "хорошее": "жилое состояние",
    "среднее": "жилое состояние",
    "под самоотделку (псо)": "без ремонта (ПСО)",
    "не достроено": "без ремонта (ПСО)",
}
for d in (train, test):
    d["condition_grp"] = d["condition"].map(group)

def run_nlp2(label, class_weight=None):
    tr = train.dropna(subset=[label]); te = test.dropna(subset=[label])
    clf = Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                                  min_df=3, sublinear_tf=True, max_features=200000)),
        ("lr", LogisticRegression(max_iter=2000, C=3, class_weight=class_weight, random_state=SEED)),
    ]).fit(tr["description_clean"], tr[label])
    pred = clf.predict(te["description_clean"])
    print(f"\n=== {label} (class_weight={class_weight}) ===")
    print(classification_report(te[label], pred, zero_division=0))
    proba = clf.predict_proba(te["description_clean"]).max(axis=1)
    for thr in (0.4, 0.5, 0.6, 0.7):
        m = proba >= thr
        acc = (pred[m] == te[label].values[m]).mean()
        print(f"порог {thr}: отвечаем в {m.mean():.0%} случаев, точность среди них {acc:.0%}")
    return clf

clf_offer = run_nlp2("offer_type", class_weight=None)
clf_offer_bal = run_nlp2("offer_type", class_weight="balanced")
clf_cond = run_nlp2("condition_grp", class_weight="balanced")



=== offer_type (class_weight=None) ===
                 precision    recall  f1-score   support

      от агента       0.90      0.99      0.94      1474
от собственника       0.85      0.24      0.37       219

       accuracy                           0.90      1693
      macro avg       0.88      0.62      0.66      1693
   weighted avg       0.89      0.90      0.87      1693

порог 0.4: отвечаем в 100% случаев, точность среди них 90%
порог 0.5: отвечаем в 100% случаев, точность среди них 90%
порог 0.6: отвечаем в 98% случаев, точность среди них 90%
порог 0.7: отвечаем в 84% случаев, точность среди них 94%

=== offer_type (class_weight=balanced) ===
                 precision    recall  f1-score   support

      от агента       0.95      0.86      0.90      1474
от собственника       0.42      0.69      0.52       219

       accuracy                           0.84      1693
      macro avg       0.68      0.78      0.71      1693
   weighted avg       0.88      0.84      0.85    